<a href="https://colab.research.google.com/github/DeepaJain29/Email_Spam_Detection/blob/main/EmailSpamDetection_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Email Spam Detection (Text Classification)

## Dataset:
https://archive.ics.uci.edu/ml/datasets/sms+spam+collection

## Step-by-Step Flow

1. Download dataset (SMSSpamCollection).

2. Load into Pandas (tab-separated file).

3. Preprocess text: Lowercase → remove punctuation → remove stopwords → stemming/lemmatization.

4. Feature Extraction:

- Use CountVectorizer or TfidfVectorizer from sklearn.

5. Split dataset into train/test.

6. Train models:

- Start with MultinomialNB (Naive Bayes).

- Try LogisticRegression or SVM.

7. Evaluation: Accuracy, Precision, Recall, F1-score, Confusion Matrix.

8. Optimization: Adjust max_features in vectorizer, try n-grams.

9. Deploy: A small web form where users paste text and see result.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('stopwords')
import os
import nltk
nltk.download('wordnet', download_dir='/usr/local/nltk_data')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/local/nltk_data...


In [ ]:
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])
print(df.sample(5))
print(df.columns)

     label                                            message
3720  spam  Thanks for your ringtone order, reference numb...
4213  spam  Missed call alert. These numbers called but le...
458    ham  I hope you that's the result of being consiste...
4930  spam  Got what it takes 2 take part in the WRC Rally...
4267   ham  The greatest test of courage on earth is to be...
Index(['label', 'message'], dtype='object')


In [ ]:
class Preprocessor:
  # Preprocess text: Lowercase → remove punctuation → remove stopwords → stemming/lemmatization.
    def __init__(self, df):
        self.df = df

    def lowercase(self):
        self.df['message'] = self.df['message'].str.lower()

    def remove_html_tags(self):
      df['message'] = self.df['message'].str.replace('<[^<]+?>', '')

    def remove_urls(self):
        self.df['message'] = self.df['message'].str.replace('http\S+|www.\S+', '', case=False)

    def remove_numbers(self):
        self.df['message'] = self.df['message'].str.replace('\d+', '')

    def remove_punctuation(self):
        self.df['message'] = self.df['message'].str.replace('[^\w\s]', '')

    def remove_stopwords(self):
        stop = stopwords.words('english')
        self.df['message'] = self.df['message'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

    def stemming(self):
        ps = PorterStemmer()
        self.df['message'] = self.df['message'].apply(lambda x: ' '.join([ps.stem(word) for word in x.split()]))

    def lemmatize(self):
        lemmatizer = WordNetLemmatizer()
        self.df['message'] = self.df['message'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))


In [ ]:
vectorizer = CountVectorizer(ngram_range=(2, 2),lowercase=True, stop_words='english', analyzer = 'word')
X = vectorizer.fit_transform(df['message'])
vectorizer.get_feature_names_out()

In [ ]:
preprocessor = Preprocessor(df)
preprocessor.lowercase()
preprocessor.remove_punctuation()
preprocessor.remove_stopwords()
# Choose either stemming or lemmatization, not both.
# preprocessor.stemming()
preprocessor.lemmatize()

print(df.sample(5))

     label                                            message
4324   ham                           aight well keep informed
1842   ham                                         office na.
2518   ham                                  sorry, call later
3665   ham                 huh? 6 also cannot? many mistakes?
3842   ham  howz pain.it come today.do said ystrday.ice me...


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X = vectorizer.fit_transform(df['message']).toarray()
y = df['label']

print("Shape of features (X):", X.shape)
print("Shape of labels (y):", y.shape)

Shape of features (X): (5572, 5000)
Shape of labels (y): (5572,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (4179, 5000)
Shape of X_test: (1393, 5000)
Shape of y_train: (4179,)
Shape of y_test: (1393,)


In [ ]:
import nltk
print(nltk.data.path)

['/root/nltk_data', '/usr/nltk_data', '/usr/share/nltk_data', '/usr/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']


In [ ]:
!python -m nltk.downloader -d /usr/local/nltk_data wordnet

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package wordnet to /usr/local/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train Multinomial Naive Bayes model
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

# Make predictions
y_pred = mnb.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='spam')
recall = recall_score(y_test, y_pred, pos_label='spam')
f1 = f1_score(y_test, y_pred, pos_label='spam')
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.9806
Precision: 1.0000
Recall: 0.8548
F1-score: 0.9217
Confusion Matrix:
[[1207    0]
 [  27  159]]
